In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_connection_webi as( 
    select distinct
        fl1.universe_id,
        fl1.universe_name,
        case
            when cms1.Connection_Name is not null then cms1.Connection_Name
            when unv1.Database_connection is not null then unv1.Database_connection
            else null
        end as Database_connection,
        -- count(distinct cms1.Document_Id) as report_cnt
        cms1.Document_Id
    from (
        select distinct
            universe_id,
            case 
                when universe_name like '%.unx' then replace(universe_name, '.unx', '') 
                else universe_name 
            end as universe_name
        from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_extraction_summary
    ) as fl1
    left join (
        select
            Document_Id,
            DataSource_ID,
            DataSource_Name,
            DataSource_Type,
            Connection_Name
        from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
        where DataSource_Type in ('unv', 'unx')
        and Connection_Name is not null
        qualify row_number() over (partition by Document_CUID order by ingestion_ts desc) = 1
    ) as cms1
        on trim(cms1.DataSource_ID) = trim(fl1.universe_id)
    and upper(trim(replace(fl1.universe_name, '_old_udt', ''))) = upper(trim(cms1.DataSource_Name))
    left join (
        select
            SI_ID as universe_id,
            SI_NAME as universe_name,
            Data_Connection_Name as Database_connection
        from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Universe_prod_connections
        qualify row_number() over (partition by SI_ID order by Ingestion_date desc) = 1
    ) as unv1
        on trim(unv1.universe_id) = trim(fl1.universe_id)
    and upper(trim(replace(fl1.universe_name, '_old_udt', ''))) = upper(trim(unv1.universe_name))
)

In [0]:
%sql
select
    'before' as cat,
    fl1.universe_id,
    fl1.universe_name,
    case
        when cms1.Connection_Name is not null then cms1.Connection_Name
        when unv1.Database_connection is not null then unv1.Database_connection
        else null
    end as Database_connection,
    count(distinct cms1.Document_Id) as report_cnt
from (
    select distinct
        universe_id,
        case 
            when universe_name like '%.unx' then replace(universe_name, '.unx', '') 
            else universe_name 
        end as universe_name
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_extraction_summary
) as fl1
left join (
    select
        Document_Id,
        DataSource_ID,
        DataSource_Name,
        Connection_Name
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
    where DataSource_Type in ('unv', 'unx')
    and Connection_Name is not null
    qualify row_number() over (partition by Document_CUID order by ingestion_ts desc) = 1
) as cms1
    on trim(cms1.DataSource_ID) = trim(fl1.universe_id)
and upper(trim(replace(fl1.universe_name, '_old_udt', ''))) = upper(trim(cms1.DataSource_Name))
left join (
    select
        SI_ID as universe_id,
        SI_NAME as universe_name,
        Data_Connection_Name as Database_connection
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Universe_prod_connections
    qualify row_number() over (partition by SI_ID order by Ingestion_date desc) = 1
) as unv1
    on trim(unv1.universe_id) = trim(fl1.universe_id)
and upper(trim(replace(fl1.universe_name, '_old_udt', ''))) = upper(trim(unv1.universe_name))
group by 
    fl1.universe_id, 
    fl1.universe_name, 
    case
        when cms1.Connection_Name is not null then cms1.Connection_Name
        when unv1.Database_connection is not null then unv1.Database_connection
        else null
    end
union all
select 'after' as cat, universe_id, universe_name, Database_connection,count(distinct Document_Id) as report_cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_connection_webi
group by cat, universe_id, universe_name, Database_connection
order by universe_id, cat asc

In [0]:
%sql

select distinct universe_name from dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition